### Trenowanie modelu kNN (Wyszukiwanie podobnych ofert)

In [ ]:
import pandas as pd
import numpy as np
import datetime

df = pd.read_json('../backend/data/raw/fast_cars_data.json')

def przelicz_cene(cena_str):
    if pd.isna(cena_str) or cena_str == 'Brak':
        return np.nan
    cena_str = str(cena_str).replace(' ', '') 
    if 'EUR' in cena_str:
        return int(float(cena_str.replace('EUR', '')) * 4.30)
    elif 'PLN' in cena_str:
        return int(float(cena_str.replace('PLN', '')))
    return np.nan

df['Cena'] = df['Cena'].apply(przelicz_cene)
df = df.dropna(subset=['Cena'])

df = df.replace('Brak', np.nan)
df = df.drop_duplicates(subset=['URL'])
df = df[~df['Rodzaj paliwa'].isin(['Benzyna+CNG', 'Wodór'])]
df.loc[df['Rodzaj paliwa'] == 'Elektryczny', 'Pojemność skokowa'] = '0'
df = df.dropna(subset=['Moc', 'Pojemność skokowa', 'Skrzynia biegów'])

df['Przebieg'] = df['Przebieg'].astype(str).str.replace(' km', '', regex=False).str.replace(' ', '', regex=False).astype(int)
df['Moc'] = df['Moc'].astype(str).str.replace(' KM', '', regex=False).str.replace(' ', '', regex=False).astype(int)
df['Pojemność skokowa'] = df['Pojemność skokowa'].astype(str).str.replace(' cm3', '', regex=False).str.replace(' ', '', regex=False).astype(float).astype(int)

df = df[(df['Cena'] >= 2500) & (df['Cena'] <= 800000)]
df = df[(df['Przebieg'] < 700000)]
df = df[(df['Pojemność skokowa'] == 0) | ((df['Pojemność skokowa'] >= 600) & (df['Pojemność skokowa'] <= 8500))]
df = df[df['Rok produkcji'] >= 1990]

df_ref = df.reset_index(drop=True)
print(f"Dane wyczyszczone! Gotowe ogłoszenia w bazie: {len(df_ref)}")

Dane wyczyszczone! Gotowe ogłoszenia w bazie: 14205


Feature Engineering i przygotowanie macierzy

In [ ]:
df_ref['Wiek'] = datetime.date.today().year - df_ref['Rok produkcji']
df_ref['Marka'] = df_ref['Tytuł'].str.split(' ').str[0]

czestotliwosc = df_ref['Marka'].value_counts()
popularne = czestotliwosc[czestotliwosc >= 30].index
df_ref.loc[~df_ref['Marka'].isin(popularne), 'Marka'] = 'Inne'

X_surowe = df_ref[['Przebieg', 'Pojemność skokowa', 'Moc', 'Wiek', 'Rodzaj paliwa', 'Skrzynia biegów', 'Marka']]

X = pd.get_dummies(X_surowe, drop_first=True, dtype=int)
wymiary_kolumn = X.columns

print(f"Macierz X wygenerowana. Liczba cech (kolumn): {len(wymiary_kolumn)}")

Macierz X wygenerowana. Liczba cech (kolumn): 46


Skalowanie i Trenowanie algorytmu kNN

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

scaler_knn = StandardScaler()
X_scaled = scaler_knn.fit_transform(X)

knn_model = NearestNeighbors(n_neighbors=5, metric='euclidean')
knn_model.fit(X_scaled)

print("Model kNN nauczony i gotowy do wyszukiwania!")

Model kNN nauczony i gotowy do wyszukiwania!


Funkcja wyszukująca i test na konkretnym aucie

In [ ]:
def znajdz_podobne_auta(auto_parametry, model, scaler, baza_referencyjna, sztywne_kolumny):
    nowe_auto_df = pd.DataFrame(0, index=[0], columns=sztywne_kolumny)
    
    
    for cecha, wartosc in auto_parametry.items():
        if cecha in nowe_auto_df.columns:
            nowe_auto_df.at[0, cecha] = wartosc
        elif cecha.startswith('Marka_'):
            if 'Marka_Inne' in nowe_auto_df.columns:
                nowe_auto_df.at[0, 'Marka_Inne'] = 1
                
    
    auto_scaled = scaler.transform(nowe_auto_df)
    
   
    odleglosci, indeksy = model.kneighbors(auto_scaled)
    
    print("5 NAJBARDZIEJ PODOBNYCH OFERT W BAZIE")
    for i, idx in enumerate(indeksy[0]):
        auto = baza_referencyjna.iloc[idx]
        print(f"{i+1}. {auto['Tytuł']}")
        print(f"   Cena: {auto['Cena']:,.0f} PLN | Rocznik: {auto['Rok produkcji']} | Przebieg: {auto['Przebieg']} km | Moc: {auto['Moc']} KM")
        print(f"   Link: {auto['URL']}\n")


auto_testowe = {
    'Przebieg': 275000,
    'Pojemność skokowa': 2000,
    'Moc': 114,
    'Wiek': 16, 
    'Rodzaj paliwa_Diesel': 1,
    'Skrzynia biegów_Manualna': 1,
    'Marka_Chevrolet': 1 
}

znajdz_podobne_auta(auto_testowe, knn_model, scaler_knn, df_ref, wymiary_kolumn)

=== 5 NAJBARDZIEJ PODOBNYCH OFERT W BAZIE ===
1. Chevrolet Cruze
   Cena: 7,200 PLN | Rocznik: 2010 | Przebieg: 250000 km | Moc: 125 KM
   Link: https://www.otomoto.pl/osobowe/oferta/chevrolet-cruze-OLX_ID6I8DxF.html

2. Chevrolet Captiva 2.0 d LS 5os 2WD
   Cena: 18,900 PLN | Rocznik: 2010 | Przebieg: 247000 km | Moc: 127 KM
   Link: https://www.otomoto.pl/osobowe/oferta/chevrolet-captiva-ID6HWanI.html

3. Chevrolet Captiva 2.0 d LT sport
   Cena: 3,300 PLN | Rocznik: 2011 | Przebieg: 268000 km | Moc: 150 KM
   Link: https://www.otomoto.pl/osobowe/oferta/chevrolet-captiva-ID6I8BeT.html

4. Chevrolet Orlando
   Cena: 19,500 PLN | Rocznik: 2012 | Przebieg: 239191 km | Moc: 130 KM
   Link: https://www.otomoto.pl/osobowe/oferta/chevrolet-orlando-ID6I3N53.html

5. Chevrolet Cruze 1.7 D LT+
   Cena: 14,000 PLN | Rocznik: 2013 | Przebieg: 281000 km | Moc: 130 KM
   Link: https://www.otomoto.pl/osobowe/oferta/chevrolet-cruze-ID6I8BBj.html



### Zapisanie modelu i artefaktów

In [13]:
import joblib

joblib.dump(knn_model, '../backend/data/kNN_model/knn_model_similar_cars.pkl')

joblib.dump(scaler_knn, '../backend/data/kNN_model/knn_scaler.pkl')

joblib.dump(list(wymiary_kolumn), '../backend/data/kNN_model/knn_columns.pkl')

df_ref.to_pickle('../backend/data/kNN_model/df_ref_similar_cars.pkl')

print("Wszystkie artefakty do wyszukiwania podobnych ofert zostały pomyślnie zapisane!")

Wszystkie artefakty do wyszukiwania podobnych ofert zostały pomyślnie zapisane!
